# 一、数据预处理--初步构建post, comment, user数据集

0. 属性中的非结构化数据处理

0.1 处理user

In [ ]:
import pandas as pd
import csv   # ← 新增这行

# 读入 CSV
df = pd.read_csv(f'/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/{sub_dataset}/sorted_user.csv', dtype=str)

# 将 bio 中的换行符替换为空格
df['bio'] = df['bio'].fillna('').str.replace(r'[\r\n]+', ' ', regex=True)

# （可选）再把多余的空格收敛成一个空格
df['bio'] = df['bio'].str.replace(r' {2,}', ' ', regex=True).str.strip()

# 直接写出，不传 quoting，保留默认最小引号策略
df.to_csv(f'/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/{sub_dataset}/sorted_user.csv', index=False)


0.2 处理post

In [ ]:
import pandas as pd
import csv   # ← 新增这行

# 读入 CSV
# df = pd.read_csv('/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/comment.csv', dtype=str)
df = pd.read_csv(f'/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/{sub_dataset}/comment.csv', dtype=str)

# 将 bio 中的换行符替换为空格
df['body'] = df['body'].fillna('').str.replace(r'[\r\n]+', ' ', regex=True)
df['bodywithurls'] = df['bodywithurls'].fillna('').str.replace(r'[\r\n]+', ' ', regex=True)
df['article'] = df['article'].fillna('').str.replace(r'[\r\n]+', ' ', regex=True)
df['preview'] = df['preview'].fillna('').str.replace(r'[\r\n]+', ' ', regex=True)

# （可选）再把多余的空格收敛成一个空格
df['body'] = df['body'].str.replace(r' {2,}', ' ', regex=True).str.strip()
df['bodywithurls'] = df['bodywithurls'].str.replace(r' {2,}', ' ', regex=True).str.strip()

# 直接写出，不传 quoting，保留默认最小引号策略
df.to_csv(f'/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/{sub_dataset}/comment.csv', index=False)


### 1.随后按comments属性(评论数量)对posts类型和commnets类型集合从大到小排序

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

input_dir   = '/home/wangshuo/resource/datasets/parler/csv_data/pc'
posts_dir   = '/home/wangshuo/resource/datasets/parler/csv_data/pc/posts/'
comments_dir= '/home/wangshuo/resource/datasets/parler/csv_data/pc/comments/'

os.makedirs(posts_dir, exist_ok=True)
os.makedirs(comments_dir, exist_ok=True)

csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

for csv_file in tqdm(csv_files, desc="Processing files"):
    df = pd.read_csv(os.path.join(input_dir, csv_file), dtype={'comments': str})

    # 先把 comments 列转成数值，不能转的置为 NaN，然后填 0（或你觉得更合适的值）
    df['comments'] = pd.to_numeric(df['comments'], errors='coerce').fillna(0).astype(int)

    posts_df    = df[df['datatype'] == 'posts']
    comments_df = df[df['datatype'] == 'comments']

    # 按照 comments 数值列排序
    if not posts_df.empty:
        posts_df = posts_df.sort_values(by='comments', ascending=False)
    if not comments_df.empty:
        comments_df = comments_df.sort_values(by='comments', ascending=False)

    posts_df.to_csv(os.path.join(posts_dir,   f"parler_posts_{csv_file}"),   index=False)
    comments_df.to_csv(os.path.join(comments_dir,f"parler_comments_{csv_file}"),index=False)

print("Processing and sorting completed!")


### 2.对于posts数据删除其评论comment数量少于2的post，

In [ ]:
import os
import pandas as pd
from tqdm import tqdm  # 用于显示处理进度条

# 输入和输出目录
input_dir = '/home/wangshuo/resource/datasets/parler/csv_data/pc/posts/all_posts/'
output_dir = '/home/wangshuo/resource/datasets/parler/csv_data/pc/posts/valid_posts3/'

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)

# 获取所有 CSV 文件
csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

# 遍历所有文件并处理
for csv_file in tqdm(csv_files, desc="Processing files"):
    input_file = os.path.join(input_dir, csv_file)
    output_file = os.path.join(output_dir, csv_file)
    # 读取 CSV 文件
    df = pd.read_csv(input_file)
    # 过滤条件：保留 comments >= 2 的记录
    filtered_df = df[df['comments'] >= 2]
    # 保存过滤后的数据到新文件
    if not filtered_df.empty:  # 只有非空数据才保存
        filtered_df.to_csv(output_file, index=False)

print(f"Filtered files saved to {output_dir}")


### 3.对用户数据按其评论数量和发帖数量进行排序

In [ ]:
import os
import pandas as pd

# 文件目录路径
directory_path = '/home/wangshuo/resource/DataSets/parler/csv_data/user/'

# 获取目录下所有的 CSV 文件
csv_files = [f for f in os.listdir(directory_path) if f.endswith('.csv')]

# 遍历每个 CSV 文件并排序
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)

    # 读取 CSV 文件
    df = pd.read_csv(file_path)

    # 排序：先按照 comments 降序，再按照 posts 降序
    sorted_df = df.sort_values(by=['comments', 'posts'], ascending=[False, False])

    # 将排序结果覆盖保存到原文件
    sorted_df.to_csv(file_path, index=False)

    print(f"文件 {file_path} 已按照 comments 和 posts 排序，并覆盖保存。")

print("所有文件处理完成！")


### 4.调查用户在某条件下的基数

In [ ]:
import pandas as pd

# 文件路径
input_file = '/home/wangshuo/resource/datasets/parler/csv_data/user/valid_users/parler_user000000000008.csv'

# 读取 CSV 文件
df = pd.read_csv(input_file)

# 条件过滤：comments >= 2 and human == True
# filtered_df = df[df['human'] == True]
filtered_df = df[((df['comments'] >= 2)) | (df['posts'] >= 1)]
# 统计满足条件的元组数量
filtered_count = filtered_df.shape[0]

# 文件中元组的总数量
total_count = df.shape[0]

# 输出统计结果
print(f"满足条件的元组数量: {filtered_count}")
print(f"文件总元组数量: {total_count}")


### 5.对每一个CSV文件保存满足filtered_df = df[((df['comments'] >= 2)) | (df['posts'] >= 1))] 条件的元组，并将这些CSV文件依次保存到文件夹

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 输入和输出目录
input_dir = '/home/wangshuo/resource/DataSets/parler/csv_data/user/'
output_dir = '/home/wangshuo/resource/DataSets/parler/csv_data/user/valid_users/'

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)

# 获取输入目录下所有 CSV 文件
csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

# 遍历每个 CSV 文件，添加进度条
for csv_file in tqdm(csv_files, desc="Processing CSV files"):
    input_file_path = os.path.join(input_dir, csv_file)
    output_file_path = os.path.join(output_dir, csv_file)

    # 读取 CSV 文件
    df = pd.read_csv(input_file_path)

    # 筛选满足条件的元组
    filtered_df = df[(df['comments'] >= 2) | (df['posts'] >= 1)]

    # 保存到新的目录
    filtered_df.to_csv(output_file_path, index=False)

print("所有文件处理完成！")


### 5.1 对文件夹/home/wangshuo/resource/DataSets/parler/csv_data/user/valid_users/下每一个CSV文件保存满足 
（只保留bio非空，且单词数量大于等于5个单词的元组，）并将这些CSV文件依次保存到文件夹/home/wangshuo/resource/DataSets/parler/csv_data/user/valid_users2/下

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 输入和输出目录
input_dir = '/home/wangshuo/resource/datasets/parler/csv_data/user/valid_users/'
output_dir = '/home/wangshuo/resource/datasets/parler/csv_data/user/valid_users2/'

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)

# 获取输入目录下所有 CSV 文件
csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

# 遍历每个 CSV 文件，添加进度条
for csv_file in tqdm(csv_files, desc="Processing CSV files"):
    input_file_path = os.path.join(input_dir, csv_file)
    output_file_path = os.path.join(output_dir, csv_file)

    # 读取 CSV 文件
    df = pd.read_csv(input_file_path, dtype=str)

    # 计算 bio 为空的掩码
    empty_mask = df['bio'].isna() | df['bio'].str.strip().eq('')
    # 计算每条 bio 的单词数
    word_counts = df['bio'].fillna('').str.split().apply(len)

    # 保留 bio 非空 且 单词数 >= 3
    keep_mask = (~empty_mask) & (word_counts >= 5)
    filtered_df = df[keep_mask].copy()

    # 保存到新的目录
    filtered_df.to_csv(output_file_path, index=False)

print("所有文件处理完成！")


### 5.2 /home/wangshuo/resource/datasets/parler/csv_data/pc/posts/valid_posts下面有XX个posts的文件，每个文件的creator字段是该文件中一个元组（帖子post）所关联用户的唯一标识符和user文件中的id字段，/home/wangshuo/resource/datasets/parler/csv_data/user/valid_users2/文件夹下保存的是user文件， 以这些user文件为基础依次扫描全部的posts文件，文件只保留每个post元组中中creator字段在user文件中有对应的元组，并将这些过滤后的文件保存到/home/wangshuo/resource/datasets/parler/csv_data/pc/posts/valid_posts2文件夹下

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 输入和输出目录
posts_input_dir = '/home/wangshuo/resource/datasets/parler/csv_data/pc/posts/valid_posts3'
users_input_dir = '/home/wangshuo/resource/datasets/parler/csv_data/user/valid_users2'
posts_output_dir = '/home/wangshuo/resource/datasets/parler/csv_data/pc/posts/valid_posts4'

# 确保输出目录存在
os.makedirs(posts_output_dir, exist_ok=True)

# 读取所有用户ID集合
user_ids = set()
for user_file in os.listdir(users_input_dir):
    if not user_file.endswith('.csv'):
        continue
    df_u = pd.read_csv(os.path.join(users_input_dir, user_file), usecols=['id'], dtype=str)
    user_ids.update(df_u['id'].dropna().astype(str).tolist())

# 处理每个 post 文件
for post_file in tqdm(os.listdir(posts_input_dir), desc='Filtering posts'):
    if not post_file.endswith('.csv'):
        continue
    in_path = os.path.join(posts_input_dir, post_file)
    out_path = os.path.join(posts_output_dir, post_file)

    # 读取 posts 文件，只加载 creator 列和全部列
    df_p = pd.read_csv(in_path, dtype=str)

    # 过滤：仅保留 creator 在 user_ids 集合中的记录
    df_filtered = df_p[df_p['creator'].isin(user_ids)].copy()
    print(len(df_filtered))
    # 保存结果
    df_filtered.to_csv(out_path, index=False)

print('所有 posts 文件过滤完成！')


### 6./home/wangshuo/resource/DataSets/parler/csv_data/pc/posts/valid_posts路径下有N个csv文件，随机抽取n个CSV文件，
每个CSV文件取前m个元组，共n*m个元组，按属性comments的大小降序排序，然后保存到/home/wangshuo/resource/DataSets/parler/csv_data/pc/posts/sub_posts下的post.csv文件中

In [ ]:
import os
import random
import pandas as pd

random_file_num = 90
tuple_num = 1000
# 设置路径
valid_posts_path = '/home/wangshuo/resource/datasets/parler/csv_data/pc/posts/valid_posts4'
sub_posts_path = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset'
# sub_posts_path = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/little_dataset'
output_file = os.path.join(sub_posts_path, 'post.csv')

# 获取valid_posts目录下的所有CSV文件
csv_files = [f for f in os.listdir(valid_posts_path) if f.endswith('.csv')]

# 随机抽取10个CSV文件
random_files = random.sample(csv_files, random_file_num)

# 存储所有抽取的数据
all_data = []

# 读取每个CSV文件的前500行
for file in random_files:
    file_path = os.path.join(valid_posts_path, file)
    df = pd.read_csv(file_path)
    all_data.append(df.head(tuple_num))  # 获取前500个元组

# 合并所有数据
combined_df = pd.concat(all_data)

# 按照comments列的大小降序排序
sorted_df = combined_df.sort_values(by='comments', ascending=False)

# 保存到输出文件
sorted_df.to_csv(output_file, index=False)
print(len(sorted_df))
print(f"数据已成功保存到 {output_file}")


### 7.在文件夹/home/wangshuo/resource/DataSets/parler/csv_data/pc/comments下面有N个comment的文件，其中post字段是该文件中一个元组（评论comment）所关联帖子的唯一标识符，
我抽取的n*m个评论数量多的post所组成的csv的数据结构和一个样例：其中id是当前post的唯一标识符，利用这个id和上面comment的post属性值即可确定哪个评论comment是id为post帖子的评论。
接下来我要统计这n*m个post每个post在这N个文件中总共有多少和其相关联的comment数量，这个数量打印出来，并作为sub_comment属性保存到n*m个post中

In [ ]:
import os
import pandas as pd
from tqdm import tqdm  # 导入tqdm库

# 设置路径
comments_path = '/home/wangshuo/resource/DataSets/parler/csv_data/pc/comments'
sub_posts_path = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data'
output_file = os.path.join(sub_posts_path, 'post_test_with_sub_comment.csv')

# 读取已抽取的5000个评论文件
post_test_file = os.path.join(sub_posts_path, 'post_test.csv')
post_test_df = pd.read_csv(post_test_file)

# 获取所有评论文件
comment_files = [f for f in os.listdir(comments_path) if f.endswith('.csv')]

# 初始化一个字典来统计每个post的评论数量
post_comment_count = {post_id: 0 for post_id in post_test_df['id']}  # 初始化为5000个post的字典
# 
# 使用tqdm显示文件处理的进度
for file in tqdm(comment_files, desc="Processing comment files"):
    file_path = os.path.join(comments_path, file)
    df = pd.read_csv(file_path)

    # 遍历该文件的每一行，只统计post_test_df中存在的post
    for post_id in df['post']:
        if post_id in post_comment_count:
            post_comment_count[post_id] += 1

# 为post_test_df中的每个评论添加sub_comment字段
post_test_df['sub_comment'] = post_test_df['id'].apply(lambda post_id: post_comment_count.get(post_id, 0))

# 保存结果到新文件
post_test_df.to_csv(output_file, index=False)

print(f"数据已成功保存到 {output_file}")


In [ ]:
post_comment_count
post_test_df['sub_comment'] = post_test_df['id'].apply(lambda post_id: post_comment_count.get(post_id, 0))
# 保存结果到新文件
post_test_df.to_csv(output_file, index=False)

### 8.将上面与post关联的所有comment保存到文件夹/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/comment.csv中

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 路径配置
comments_dir = '/home/wangshuo/resource/datasets/parler/csv_data/pc/comments/valid_comments2'
sub_posts_dir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset'
# sub_posts_dir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/little_dataset'
os.makedirs(sub_posts_dir, exist_ok=True)
comment_output_file = os.path.join(sub_posts_dir, 'comment.csv')

# 读取目标 posts 列表
post_test_file = os.path.join(sub_posts_dir, 'post.csv')
post_df = pd.read_csv(post_test_file, dtype=str)
post_ids = set(post_df['id'].astype(str))

# 用于保存所有关联到 post 的评论
relevant_comments = []
# 用于记录每个评论文件的匹配数量
file_match_counts = {}

# 遍历所有评论文件
for fname in tqdm(os.listdir(comments_dir), desc="Processing comment files"):
    if not fname.endswith('.csv'):
        continue
    path = os.path.join(comments_dir, fname)
    df = pd.read_csv(path, dtype=str)

    # 过滤：只保留 post 字段在 post_ids 中的行
    mask = df['post'].isin(post_ids)
    matched_df = df[mask]
    
    # 记录数量
    match_count = len(matched_df)
    file_match_counts[fname] = match_count
    print(f"{fname}: {match_count} 条评论关联到 post.csv 中的帖子")

    # 收集所有匹配到的评论
    if match_count:
        relevant_comments.append(matched_df)

# 打印汇总
total_matched = sum(file_match_counts.values())
print("\n—— 汇总 ——")
for fname, cnt in file_match_counts.items():
    print(f"  {fname:30s}: {cnt:4d}")
print(f"所有文件累计：{total_matched} 条评论关联到 posts")

# 合并并保存到一个 CSV
if relevant_comments:
    result_df = pd.concat(relevant_comments, ignore_index=True)
    result_df.to_csv(comment_output_file, index=False)
    print(f"\n所有关联评论已保存到：{comment_output_file}")
else:
    print("\n未找到任何关联评论。")


### 8.1 评论下一级评论 comment文件中parent属性表示当前评论的父级评论ID，/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/下存有comment.csv 这个文件中的评论作为父级评论查找/home/wangshuo/resource/datasets/parler/csv_data/pc/comments/valid_comments2 文件夹下存有的各种commentx.csv文件中的子评论

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 路径配置
parent_file   = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/comment.csv'
children_dir  = '/home/wangshuo/resource/datasets/parler/csv_data/pc/comments/valid_comments2'
output_file   = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/comment_child.csv'

# 1. 读取父级评论 ID 列表
parent_df = pd.read_csv(parent_file, dtype=str, usecols=['id'])
parent_ids = set(parent_df['id'].dropna())

# 用于统计
file_counts = {}
per_parent_counts = {pid: 0 for pid in parent_ids}

# 收集所有匹配到的子评论
matched_children = []

# 2. 遍历所有子评论文件
for fname in tqdm(os.listdir(children_dir), desc="Scanning child files"):
    if not fname.endswith('.csv'):
        continue
    path = os.path.join(children_dir, fname)
    df = pd.read_csv(path, dtype=str, usecols=['parent', 'id', 'body', 'creator', 'createdAt'])

    # 3. 筛选 parent 在父级列表中的行
    mask = df['parent'].isin(parent_ids)
    matched = df[mask]
    count = len(matched)
    file_counts[fname] = count

    # 统计每个父级评论被匹配到的子评论数
    for pid, grp in matched.groupby('parent'):
        per_parent_counts[pid] += len(grp)

    # 收集
    if count:
        matched_children.append(matched)

    print(f"{fname}: 找到 {count} 条子评论")

# 合并所有并保存
if matched_children:
    result = pd.concat(matched_children, ignore_index=True)
    # 4. 写出完整子评论表
    result.to_csv(output_file, index=False)
    print(f"\n所有子评论已保存到：{output_file}")
else:
    print("\n未找到任何子评论。")

# 最终汇总
total = sum(file_counts.values())
print("\n—— 文件匹配汇总 ——")
for fname, cnt in file_counts.items():
    print(f"  {fname:30s}: {cnt:4d}")

print(f"\n所有文件累计匹配到 {total} 条子评论。")

# # （可选）各父级评论的子评论数
# print("\n—— 每个父级评论的子评论数 ——")
# for pid, cnt in per_parent_counts.items():
#     if cnt > 0:
#         print(f"  父评论 {pid}: {cnt} 条子评论")


### 8.2 将comment_child.csv的文件内容追加到comment.csv文件 后面

In [ ]:
import pandas as pd

base_path  = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/75W_valid_user'
parent_fp  = f'{base_path}/comment.csv'
child_fp   = f'{base_path}/comment_child.csv'

# 读取两个文件
df_parent = pd.read_csv(parent_fp, dtype=str)
df_child  = pd.read_csv(child_fp,  dtype=str)

# 拼接
df_all = pd.concat([df_parent, df_child], ignore_index=True)


# 覆写回原 comment.csv
df_all.to_csv(parent_fp, index=False)


### 8.3 comment文件中parent属性表示当前评论的父级评论ID，统计comment.csv中有多少评论的parent属性在comment.csv的id属性中有对应的元组

In [6]:
import pandas as pd
data_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'

# 读取 CSV
df = pd.read_csv(data_dir + "comment.csv")

# 评论总数
total_comments = len(df)

# id 集合
id_set = set(df["id:ID"])

# parent 存在于 id 集合中的评论
has_parent = df["parent"].isin(id_set).sum()

# parent 不存在于 id 集合中的评论
no_parent = total_comments - has_parent

print("评论总数量:", total_comments)
print("在文件内能找到父评论的评论数量:", has_parent)
print("在文件内找不到父评论的评论数量:", no_parent)




评论总数量: 131268
在文件内能找到父评论的评论数量: 17755
在文件内找不到父评论的评论数量: 113513


### 8.4 将comment.csv文件中能找到父评论的评论保存到comment_valid_child.csv文件中，然后从原文件删除，原文件只保存找不父评论的评论

In [10]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
# 读取原始文件
df = pd.read_csv(data_dir + "comment.csv")

# 所有评论的 id 集合
id_set = set(df["id:ID"])

# 能找到父评论的评论
valid_child = df[df["parent"].isin(id_set)]

# 不能找到父评论的评论
orphan = df[~df["parent"].isin(id_set)]

# 保存到新文件
valid_child.to_csv(data_dir + "comment_valid_child.csv", index=False, encoding="utf-8")

# 覆盖原文件（只保存找不到父评论的评论）
orphan.to_csv(data_dir + "comment_of_post.csv", index=False, encoding="utf-8")

print("有效子评论数量:", len(valid_child))
print("孤立评论数量:", len(orphan))
print("处理完成，comment_valid_child.csv 保存了有效子评论，comments.csv 更新为仅孤立评论")


有效子评论数量: 17755
孤立评论数量: 113513
处理完成，comment_valid_child.csv 保存了有效子评论，comments.csv 更新为仅孤立评论


### 9.接下来分别读取post.csv，comment.csv，这两个文件中均有属性creator, creator对应user中的id属性，
为每一个post.csv，comment.csv元组中出现的creator，保存在user中对应的元组，
元组是唯一的，并将这些元组保存到users.csv中，打印users元组的数量。

In [ ]:
import os
import pandas as pd
from tqdm import tqdm  # 导入tqdm库

# 设置路径
valid_users_path = '/home/wangshuo/resource/datasets/parler/csv_data/user/valid_users2'
post_test_path = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/post.csv'
comment_test_path = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/comment.csv'
output_file = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/user.csv'

# 读取所有用户文件并合并为一个DataFrame
user_files = [f for f in os.listdir(valid_users_path) if f.endswith('.csv')]
user_dataframes = []

print("开始读取所有用户数据...")

# 使用tqdm显示文件处理的进度
for file in tqdm(user_files, desc="读取用户文件"):
    file_path = os.path.join(valid_users_path, file)
    df = pd.read_csv(file_path)
    user_dataframes.append(df)

# 合并所有用户数据
users_df = pd.concat(user_dataframes, ignore_index=True)

print(f"所有用户数据已合并，共包含 {len(users_df)} 条记录。")

# 读取post_test.csv和comment_test.csv
print("开始读取 post_test.csv 和 comment_test.csv...")
post_test_df = pd.read_csv(post_test_path)
comment_test_df = pd.read_csv(comment_test_path)

print(f"post_test.csv 包含 {len(post_test_df)} 条记录，comment_test.csv 包含 {len(comment_test_df)} 条记录。")

# 创建一个集合来保存唯一的creator id
unique_creators = set()

# 将post_test中的creator添加到unique_creators集合
print("从post_test.csv提取唯一的creator ID...")
unique_creators.update(post_test_df['creator'].unique())

# 将comment_test中的creator添加到unique_creators集合
print("从comment_test.csv提取唯一的creator ID...")
unique_creators.update(comment_test_df['creator'].unique())

print(f"共找到 {len(unique_creators)} 个唯一的creator ID。")

# 过滤出users_df中creator id对应的用户信息
print("开始从用户数据中筛选相关用户...")
users_test_df = users_df[users_df['id'].isin(unique_creators)]

# 记录没有找到的用户 id
missing_users = users_df[~users_df['id'].isin(unique_creators)]

# 保存到users_test.csv
print("将筛选后的用户数据保存到 users_test.csv...")
users_test_df.to_csv(output_file, index=False)

# 打印用户元组的数量


In [ ]:
print(f"用户元组的数量: {len(users_test_df)}")

# 输出没有找到对应 post 或 comment 的用户 ID 和数量
if not missing_users.empty:
    print(f"未找到对应 post 或 comment 的用户 ID 总数: {len(missing_users)}")
    print("未找到的用户 ID 列表:")
    print(missing_users['id'].tolist())
else:
    print("所有用户都找到了对应的 post 或 comment。")
